# 02 — Load Static GTFS

Finds the most recent downloaded snapshot under `~/transit-ai-data/gtfs_static/`,
loads the key GTFS txt files into DataFrames, casts column types, and writes
Parquet files to `~/transit-ai-data/gtfs_static/parquet/` for fast EDA loading.

In [1]:
# Cell 0 — Environment setup: load .env, configure S3 access
import os
import subprocess
import sys
from pathlib import Path

try:
    from dotenv import load_dotenv, find_dotenv
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'python-dotenv', '-q'], check=True)
    from dotenv import load_dotenv, find_dotenv

try:
    import s3fs
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 's3fs', '-q'], check=True)
    import s3fs

_dotenv_path = find_dotenv(usecwd=True)
if _dotenv_path:
    load_dotenv(_dotenv_path, override=False)
    print(f'Loaded .env from: {_dotenv_path}')
else:
    print('No .env file found — using defaults.')

S3_BUCKET = os.environ.get('AWS_S3_BUCKET', '')
AWS_REGION = os.environ.get('AWS_REGION', 'ap-southeast-2')
REPO_DIR   = os.environ.get('TRANSIT_AI_REPO_DIR', str(Path.cwd().parent))

if not S3_BUCKET:
    raise EnvironmentError('AWS_S3_BUCKET is not set. Check your .env file.')

fs = s3fs.S3FileSystem()

print(f'S3_BUCKET: {S3_BUCKET}')
print(f'AWS_REGION: {AWS_REGION}')
print(f'REPO_DIR: {REPO_DIR}')

Loaded .env from: /Users/proteeksanyal/Desktop/Learning/Transit-AI/.env
S3_BUCKET: seq-transit-ai-data-ps
AWS_REGION: ap-southeast-2
REPO_DIR: /Users/proteeksanyal/Desktop/Learning/Transit-AI


In [2]:
# Cell 1 — Locate the most recent GTFS snapshot in S3
import re

STATIC_PREFIX  = f'{S3_BUCKET}/gtfs_static'
PARQUET_PREFIX = f'{S3_BUCKET}/gtfs_static/parquet'

date_pattern = re.compile(r'^\d{4}-\d{2}-\d{2}$')

try:
    all_entries = fs.ls(STATIC_PREFIX)
except Exception as e:
    raise FileNotFoundError(f'Cannot list s3://{STATIC_PREFIX}: {e}')

date_dirs = sorted([
    e for e in all_entries
    if date_pattern.match(e.rstrip('/').split('/')[-1])
])

if not date_dirs:
    raise FileNotFoundError(
        f'No YYYY-MM-DD subfolders found under s3://{STATIC_PREFIX}.\n'
        'Run scripts/archive_gtfsrt.py (or the download cell in notebook 01) first.'
    )

GTFS_PREFIX = date_dirs[-1]
print(f'Using snapshot: s3://{GTFS_PREFIX}')
txt_files = [f.split('/')[-1] for f in fs.ls(GTFS_PREFIX) if f.endswith('.txt')]
print('Files present:', sorted(txt_files))

Using snapshot: s3://seq-transit-ai-data-ps/gtfs_static/2026-07-01
Files present: ['agency.txt', 'calendar.txt', 'calendar_dates.txt', 'feed_info.txt', 'routes.txt', 'shapes.txt', 'stop_times.txt', 'stops.txt', 'trips.txt']


In [3]:
# Cell 2 — Load GTFS txt files from S3 into DataFrames
import pandas as pd

def load_txt(name, **kwargs):
    s3_path = f's3://{GTFS_PREFIX}/{name}'
    try:
        return pd.read_csv(s3_path, dtype=str, **kwargs)
    except FileNotFoundError:
        print(f'  {name} not found — skipping')
        return None

stops          = load_txt('stops.txt')
routes         = load_txt('routes.txt')
trips          = load_txt('trips.txt')
stop_times     = load_txt('stop_times.txt')
calendar       = load_txt('calendar.txt')
calendar_dates = load_txt('calendar_dates.txt')
shapes         = load_txt('shapes.txt')

frames = {
    'stops': stops,
    'routes': routes,
    'trips': trips,
    'stop_times': stop_times,
    'calendar': calendar,
    'calendar_dates': calendar_dates,
    'shapes': shapes,
}

print('\nLoaded:')
for name, df in frames.items():
    if df is not None:
        print(f'  {name:>15}: {len(df):>10,} rows × {df.shape[1]} cols')


Loaded:
            stops:     13,081 rows × 11 cols
           routes:      1,281 rows × 8 cols
            trips:    139,413 rows × 7 cols
       stop_times:  3,509,446 rows × 7 cols
         calendar:        174 rows × 10 cols
   calendar_dates:         32 rows × 3 cols
           shapes:  1,044,755 rows × 4 cols


In [4]:
# Cell 3 — Shape and dtypes sanity check
for name, df in frames.items():
    if df is None:
        continue
    print(f'\n=== {name} ({df.shape[0]:,} rows \u00d7 {df.shape[1]} cols) ===')
    print(df.dtypes.to_string())
    print(df.head(2).to_string(index=False))


=== stops (13,081 rows × 11 cols) ===
stop_id           object
stop_code         object
stop_name         object
stop_desc         object
stop_lat          object
stop_lon          object
zone_id           object
stop_url          object
location_type     object
parent_station    object
platform_code     object
stop_id stop_code                                stop_name stop_desc    stop_lat    stop_lon zone_id                                   stop_url location_type parent_station platform_code
      1    000001   Herschel Street Stop 1 near North Quay       NaN  -27.467834  153.019079       1 https://translink.com.au/stop/000001/gtfs/             0            NaN           NaN
     10    000010 Ann Street Stop 11 at King George Square       NaN  -27.468003  153.023970       1 https://translink.com.au/stop/000010/gtfs/             0            NaN           NaN

=== routes (1,281 rows × 8 cols) ===
route_id            object
route_short_name    object
route_long_name     object
route_

In [5]:
# Cell 4 — Cast columns to correct types
#
# stop_lat / stop_lon: str -> float
# departure_time / arrival_time: kept as str -- GTFS allows 25:xx overflow times
# that cannot be represented by datetime; string ordering works for sequencing.
# shape_pt_lat / shape_pt_lon: str -> float
# shape_pt_sequence: str -> Int64
# stop_sequence: str -> Int64

if stops is not None:
    stops['stop_lat'] = pd.to_numeric(stops['stop_lat'], errors='coerce')
    stops['stop_lon'] = pd.to_numeric(stops['stop_lon'], errors='coerce')

if shapes is not None:
    shapes['shape_pt_lat']      = pd.to_numeric(shapes['shape_pt_lat'],      errors='coerce')
    shapes['shape_pt_lon']      = pd.to_numeric(shapes['shape_pt_lon'],      errors='coerce')
    shapes['shape_pt_sequence'] = pd.to_numeric(shapes['shape_pt_sequence'], errors='coerce').astype('Int64')
    if 'shape_dist_traveled' in shapes.columns:
        shapes['shape_dist_traveled'] = pd.to_numeric(shapes['shape_dist_traveled'], errors='coerce')

if stop_times is not None and 'stop_sequence' in stop_times.columns:
    stop_times['stop_sequence'] = pd.to_numeric(stop_times['stop_sequence'], errors='coerce').astype('Int64')

print('Type casting complete.')
if stops is not None:
    print(f'  stops lat/lon: {stops["stop_lat"].dtype} / {stops["stop_lon"].dtype}')
if stop_times is not None:
    print(f'  stop_times departure_time: {stop_times["departure_time"].dtype} (kept as string)')

Type casting complete.
  stops lat/lon: float64 / float64
  stop_times departure_time: object (kept as string)


In [6]:
# Cell 5 — Save each DataFrame as Parquet to S3
saved = []
for name, df in frames.items():
    if df is None:
        continue
    s3_path = f's3://{PARQUET_PREFIX}/{name}.parquet'
    df.to_parquet(s3_path, index=False)
    saved.append((name, len(df), s3_path))
    print(f'  Saved {name}.parquet ({len(df):,} rows) -> {s3_path}')

print(f'\n{len(saved)} Parquet files written to s3://{PARQUET_PREFIX}')
print('Next step: run notebooks/03_load_performance_data.ipynb')

  Saved stops.parquet (13,081 rows) -> s3://seq-transit-ai-data-ps/gtfs_static/parquet/stops.parquet


  Saved routes.parquet (1,281 rows) -> s3://seq-transit-ai-data-ps/gtfs_static/parquet/routes.parquet


  Saved trips.parquet (139,413 rows) -> s3://seq-transit-ai-data-ps/gtfs_static/parquet/trips.parquet


  Saved stop_times.parquet (3,509,446 rows) -> s3://seq-transit-ai-data-ps/gtfs_static/parquet/stop_times.parquet


  Saved calendar.parquet (174 rows) -> s3://seq-transit-ai-data-ps/gtfs_static/parquet/calendar.parquet


  Saved calendar_dates.parquet (32 rows) -> s3://seq-transit-ai-data-ps/gtfs_static/parquet/calendar_dates.parquet


  Saved shapes.parquet (1,044,755 rows) -> s3://seq-transit-ai-data-ps/gtfs_static/parquet/shapes.parquet

7 Parquet files written to s3://seq-transit-ai-data-ps/gtfs_static/parquet
Next step: run notebooks/03_load_performance_data.ipynb


In [7]:
# Cell 6 — Summary
def count(df):
    return len(df) if df is not None else 0

print(
    f'Loaded {count(routes):,} routes, '
    f'{count(stops):,} stops, '
    f'{count(trips):,} trips, '
    f'{count(stop_times):,} stop_time records'
    + (f', {count(shapes):,} shape points' if shapes is not None else '')
)
print('Next step: run notebooks/03_load_performance_data.ipynb')

Loaded 1,281 routes, 13,081 stops, 139,413 trips, 3,509,446 stop_time records, 1,044,755 shape points
Next step: run notebooks/03_load_performance_data.ipynb
